# MuJoCo 環境構築（Google Colab 用）

このノートブックは **Colab 上で MuJoCo の環境構築〜動作確認まで**を行います。
Docker は Colab では使えないため、再現性は「**pip のバージョン固定**」で確保し、
描画は Colab の GPU を使った **EGL（ハードウェア・ヘッドレス描画）**で行います。

## 前提
1. メニュー **[ランタイム] → [ランタイムのタイプを変更]** で、
   **ハードウェア アクセラレータ = GPU** を選んでおく（EGL 描画に使用）。
2. 上から順にセルを実行する。

## 出典（根拠）
- Colab の EGL 設定（ICD 設定・`MUJOCO_GL=egl`）は公式チュートリアル Notebook に準拠:
  https://colab.research.google.com/github/google-deepmind/mujoco/blob/main/python/tutorial.ipynb
- MuJoCo Python バインディング（`mujoco` パッケージにライブラリ本体が同梱）:
  https://mujoco.readthedocs.io/en/stable/python.html

## 1. GPU の確認

EGL でのハードウェア描画には GPU が必要です。まず GPU が割り当てられているか確認します。

In [ ]:
import subprocess

gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
if gpu.returncode == 0:
    print("GPU 利用可能です（EGL でハードウェア描画できます）。")
    # 1行目（ドライバ情報）だけ表示
    head = gpu.stdout.strip().splitlines()
    if head:
        print(head[0])
else:
    print("GPU が見つかりません。")
    print("[ランタイム] → [ランタイムのタイプを変更] → ハードウェア アクセラレータ = GPU に設定してください。")
    print("GPU 無しで進めたい場合は、最後の『補足：osmesa フォールバック』の修正点を使ってください。")

## 2. MuJoCo のインストール（バージョン固定）

Colab ではイメージごと固定する Docker が使えないため、**パッケージのバージョンを固定**して再現性を確保します。
`mujoco==3.5.0` は 2026/02/13 に公開された版です（より新しい版は MuJoCo のリリースページで確認できます）。
MuJoCo ライブラリ本体は wheel に同梱されるので、別途ダウンロードは不要です。

In [ ]:
!pip install -q mujoco==3.5.0

import sys
print("Python:", sys.version.split()[0])
# 参考: MuJoCo は Python 3.13 以下向けにビルド済み wheel が配布されています。

## 3. EGL（GPU・ヘッドレス描画）の設定

Colab のカーネルには NVIDIA EGL ドライバの ICD 設定が無いため、自分で書き込みます。
そのうえで `MUJOCO_GL=egl` を設定し、MuJoCo の描画を EGL バックエンドに切り替えます。

> **重要**: `MUJOCO_GL` は **`import mujoco` より前**に設定する必要があります（描画コンテキスト生成時に参照されるため）。

In [ ]:
import os

# NVIDIA の EGL ベンダ設定（ICD）を書き込む（公式チュートリアルと同じ内容）
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

# 描画バックエンドを EGL に設定（import mujoco より前）
os.environ["MUJOCO_GL"] = "egl"
print("MUJOCO_GL =", os.environ["MUJOCO_GL"])

## 4. 動作確認（import → コンパイル → 1 フレーム描画）

公式チュートリアルの確認手順に沿って、(1) import、(2) 最小モデル `<mujoco/>` のコンパイル、
(3) オフスクリーンで 1 フレーム描画、を行います。最後に画像が表示されれば EGL 描画まで成功です。

In [ ]:
import mujoco
import matplotlib.pyplot as plt

print("MuJoCo   :", mujoco.__version__)
print("MUJOCO_GL:", __import__("os").environ.get("MUJOCO_GL"))

# (2) 最小の有効な MJCF モデルをコンパイル
mujoco.MjModel.from_xml_string("<mujoco/>")
print("OK: 空のモデルをコンパイルできました。")

# 真っ黒にならないようライトを入れた小さなシーン
xml = """
<mujoco>
  <worldbody>
    <light name="top" pos="0 0 1"/>
    <geom name="red_box" type="box" size=".2 .2 .2" rgba="1 0 0 1"/>
    <geom name="green_sphere" pos=".2 .2 .2" size=".1" rgba="0 1 0 1"/>
  </worldbody>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(xml)
data = mujoco.MjData(model)

# (3) オフスクリーン描画。EGL バックエンドが壊れているとここで例外になる。
with mujoco.Renderer(model) as renderer:
    mujoco.mj_forward(model, data)   # 描画前に派生量（Cartesian 座標など）を伝播：必須
    renderer.update_scene(data)
    pixels = renderer.render()

print("OK: shape", pixels.shape, "のフレームを描画しました。")
plt.imshow(pixels)
plt.axis("off")
plt.show()
print("\nMuJoCo の環境構築が完了しました。")

## 補足：GPU を使わない場合の修正点（osmesa フォールバック）

GPU ランタイムが使えない場合は、CPU ソフトウェア描画（OSMesa）に切り替えます。**変更点は次の 3 つ**です。

1. 先頭付近に OSMesa を導入するセルを追加:
   ```python
   !apt-get -qq install -y libosmesa6-dev
   ```
2. 「3. EGL の設定」セルの環境変数を変更:
   ```python
   os.environ["MUJOCO_GL"] = "osmesa"   # "egl" → "osmesa"
   ```
3. ICD（`10_nvidia.json`）の書き込みは **不要**（削除して可）。

CPU 描画なので遅くなりますが、GPU 無しでも動作します。

## 次のステップ
このノートブックは「環境構築まで」です。シミュレーションや動画出力に進む場合は、
`mediapy`（`!pip install -q mediapy`、ffmpeg は Colab に同梱）を入れ、
公式チュートリアルの `mj_step` ループ + `media.show_video(...)` を使うのが簡単です。